<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../2.%20working_with_data/7.%20database_migrations_with_flask_migrate.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 7: Migrations</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='9.%20configuration_management.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 9: Configuration →</a>
</div>

# Chapter 8: Blueprints and the Application Factory

**Learning goals:**
- Understand *why* blueprints exist and what problem they solve
- Create a blueprint, register routes on it, and mount it in a factory
- Use `url_for` correctly with blueprint namespaces
- Implement the `extensions.py` pattern for shared extensions
- Recognise the three most common beginner mistakes — and how to avoid them

## 🗺️ The Big Picture

You started with one file: **`app.py`**. It worked. But now you have routes for users, posts,
comments, authentication, and admin — all jumbled in one **1 000-line file**.
Blueprints split that file into logical modules. The **Application Factory** creates the app
in a way that makes it **testable and configurable**.

```
Before blueprints          After blueprints
──────────────────         ──────────────────────────────────
app.py (1 000 lines)  →   app/
                              __init__.py  (factory, ~30 lines)
                              auth/
                                  routes.py
                              blog/
                                  routes.py
                              admin/
                                  routes.py
```

Two ideas work together:

| Concept | Role |
|---|---|
| **Blueprint** | A *mini-app* that groups related routes, templates, and static files |
| **Application Factory** | A function (`create_app`) that builds and returns the Flask app, letting you pass in different configs |

## 💡 Core Concepts — The Why

### Blueprint analogy

Think of a large company:

- **HR department** (auth blueprint) → hiring / firing / payroll procedures
- **Finance department** (blog blueprint) → invoice / budget procedures
- **IT department** (admin blueprint) → server / access procedures

Each department runs independently with its own rulebook.  
On day one everyone registers with **company HQ** (the Flask app).  
No department needs to know what the others are doing.

A **Blueprint** is exactly that departmental instruction manual:  
it defines routes, error handlers, template filters — completely self-contained.  
The `create_app` factory is **company HQ**: it hires every department at startup.

### Application Factory analogy

Without a factory you build the app at *import time*:

```python
app = Flask(__name__)   # built immediately — untestable, config is fixed
```

With a factory you build it *on demand*:

```python
def create_app(config="production"):
    app = Flask(__name__)
    # apply config, register blueprints, init extensions …
    return app
```

Now tests can call `create_app("testing")` and production calls `create_app("production")` —
same code, different behaviour.

In [ ]:
# ── Syntax & First Usage ──────────────────────────────────────────────
# The three lines that define a blueprint:

from flask import Blueprint

# 1. Create the blueprint object
auth_bp = Blueprint(
    "auth",          # blueprint name (used in url_for: "auth.login")
    __name__,        # helps Flask locate templates / static files
    url_prefix="/auth",
)

# 2. Attach a route
@auth_bp.route("/login")
def login():
    return "Login page"

# 3. Register it (normally done inside create_app)
from flask import Flask
app = Flask(__name__)
app.register_blueprint(auth_bp)

# Verify the routes are registered
with app.app_context():
    rules = [str(r) for r in app.url_map.iter_rules()]
    for r in sorted(rules):
        print(r)


In [ ]:
# ── The Problem: Monolithic app.py ────────────────────────────────────
# Imagine every feature crammed into one file.
# Let's print what that looks like (abbreviated).

monolith = """
# app.py  (1 000 + lines — real projects get here fast)

from flask import Flask, request, jsonify, render_template, redirect, url_for
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, login_required, current_user
import hashlib, datetime, os, re

app = Flask(__name__)
db  = SQLAlchemy(app)

# ── AUTH routes (lines 30-180) ──────────────────────────────────────
@app.route("/login",  methods=["GET","POST"])
def login(): ...

@app.route("/logout")
def logout(): ...

@app.route("/register", methods=["GET","POST"])
def register(): ...

# ── BLOG routes (lines 181-420) ─────────────────────────────────────
@app.route("/")
def index(): ...

@app.route("/post/<int:id>")
def post_detail(id): ...

@app.route("/post/new", methods=["GET","POST"])
def new_post(): ...

# ── ADMIN routes (lines 421-700) ────────────────────────────────────
@app.route("/admin/dashboard")
@login_required
def admin_dashboard(): ...

@app.route("/admin/users")
@login_required
def admin_users(): ...

# ── COMMENT routes (lines 701-900) ──────────────────────────────────
@app.route("/comment/new", methods=["POST"])
def new_comment(): ...

# ── API routes (lines 901-1050) ─────────────────────────────────────
@app.route("/api/posts")
def api_posts(): ...

# … and it keeps growing …
"""

problems = [
    "❌  Git blame is useless — 5 devs editing the same file simultaneously",
    "❌  One syntax error anywhere crashes the whole app",
    "❌  Can't test auth without loading blog, admin, api, comments …",
    "❌  url_for('login') and url_for('index') — flat namespace, easy to clash",
    "❌  Can't swap in a 'testing' config without monkey-patching globals",
]

print("=== MONOLITHIC app.py ===")
print(monolith)
print("=== PROBLEMS ===")
for p in problems:
    print(p)


In [ ]:
# ── Creating a Blueprint ──────────────────────────────────────────────
# Convention: blueprints live in their own package (folder).
# Here we simulate that with plain Python objects.

from flask import Blueprint, Flask

# ── auth/routes.py (simulated) ────────────────────────────────────────
auth_bp = Blueprint("auth", __name__, url_prefix="/auth")

@auth_bp.route("/login",    methods=["GET", "POST"])
def login():    return "Auth › login"

@auth_bp.route("/logout")
def logout():   return "Auth › logout"

@auth_bp.route("/register", methods=["GET", "POST"])
def register(): return "Auth › register"

# ── blog/routes.py (simulated) ────────────────────────────────────────
blog_bp = Blueprint("blog", __name__, url_prefix="/blog")

@blog_bp.route("/")
def index():                return "Blog › index"

@blog_bp.route("/<int:id>")
def post_detail(id):        return f"Blog › post #{id}"

@blog_bp.route("/new", methods=["GET", "POST"])
def new_post():             return "Blog › new post"

# ── admin/routes.py (simulated) ───────────────────────────────────────
admin_bp = Blueprint("admin", __name__, url_prefix="/admin")

@admin_bp.route("/")
def dashboard(): return "Admin › dashboard"

@admin_bp.route("/users")
def users():     return "Admin › users"

print("✅  Three blueprints created:")
for bp in (auth_bp, blog_bp, admin_bp):
    print(f"   Blueprint(name={bp.name!r}, url_prefix={bp.url_prefix!r})")


In [ ]:
# ── Adding routes + live test with Flask test client ──────────────────
from flask import Blueprint, Flask

orders_bp = Blueprint("orders", __name__, url_prefix="/orders")

@orders_bp.route("/")
def list_orders():
    return "All orders"

@orders_bp.route("/<int:order_id>")
def order_detail(order_id):
    return f"Order #{order_id}"

@orders_bp.route("/new", methods=["POST"])
def create_order():
    return "Order created", 201

# Build a minimal app and test it
app = Flask(__name__)
app.register_blueprint(orders_bp)

client = app.test_client()

tests = [
    ("GET",  "/orders/",    200),
    ("GET",  "/orders/42",  200),
    ("POST", "/orders/new", 201),
    ("GET",  "/orders/999", 200),
]

print("=== Blueprint route tests ===")
for method, path, expected in tests:
    if method == "GET":
        resp = client.get(path)
    else:
        resp = client.post(path)
    status = "✅" if resp.status_code == expected else "❌"
    print(f"  {status}  {method:4s} {path:20s} → {resp.status_code}  body={resp.data.decode()!r}")


In [ ]:
# ── Application Factory: create_app() ────────────────────────────────
# The factory pattern: build the app inside a function so it can be
# called with different configs (production, testing, development).

from flask import Flask, Blueprint

# --- Simulated blueprints (one-liners for demo) ---
def make_blueprint(name, prefix, routes):
    bp = Blueprint(name, __name__, url_prefix=prefix)
    for path, fn_name, body in routes:
        # Attach route dynamically for demo purposes
        fn = lambda b=body: b   # noqa: E731
        fn.__name__ = fn_name
        bp.add_url_rule(path, endpoint=fn_name, view_func=fn)
    return bp

auth_bp  = make_blueprint("auth",  "/auth",  [("/login",  "login",  "login page")])
blog_bp  = make_blueprint("blog",  "/blog",  [("/",       "index",  "blog home")])
admin_bp = make_blueprint("admin", "/admin", [("/",       "dash",   "admin dash")])

# --- The factory ---
def create_app(config_name="default"):
    """Application factory — call this to get a configured Flask app."""
    app = Flask(__name__)

    # 1. Apply config (simplified)
    configs = {
        "default":    {"TESTING": False, "DEBUG": False, "SECRET_KEY": "prod-secret"},
        "testing":    {"TESTING": True,  "DEBUG": True,  "SECRET_KEY": "test-secret"},
        "development":{"TESTING": False, "DEBUG": True,  "SECRET_KEY": "dev-secret"},
    }
    app.config.update(configs.get(config_name, configs["default"]))
    app.config["CONFIG_NAME"] = config_name

    # 2. Init extensions  (would be: db.init_app(app), login_manager.init_app(app) …)
    #    — shown in the next cell

    # 3. Register blueprints
    app.register_blueprint(auth_bp)
    app.register_blueprint(blog_bp)
    app.register_blueprint(admin_bp)

    return app

# --- Demonstrate different configs ---
for cfg in ("default", "testing", "development"):
    app = create_app(cfg)
    with app.app_context():
        routes = sorted(str(r) for r in app.url_map.iter_rules()
                        if not str(r).startswith("/static"))
        print(f"\n[{cfg:11s}]  TESTING={app.config['TESTING']}  DEBUG={app.config['DEBUG']}")
        print(f"            SECRET_KEY={app.config['SECRET_KEY']!r}")
        print(f"            Routes: {routes}")


In [ ]:
# ── extensions.py pattern ────────────────────────────────────────────
# Problem: db = SQLAlchemy(app) ties the extension to ONE app instance.
# Solution: create extensions without an app, then call .init_app(app) later.

# ══ extensions.py (would be a separate file) ═════════════════════════

class FakeSQLAlchemy:
    """Minimal stand-in for flask_sqlalchemy.SQLAlchemy."""
    def __init__(self):
        self.app = None

    def init_app(self, app):
        self.app = app
        app.extensions["fake_db"] = self
        print(f"  db.init_app(app) called — bound to app {id(app):#x}")

class FakeLoginManager:
    def __init__(self):
        self.app = None
        self.login_view = None

    def init_app(self, app):
        self.app = app
        app.extensions["fake_login"] = self
        print(f"  login_manager.init_app(app) called — login_view={self.login_view!r}")

# Create extension objects at module level (no app yet)
db            = FakeSQLAlchemy()
login_manager = FakeLoginManager()
login_manager.login_view = "auth.login"   # set once here, forever

# ══ __init__.py / factory ════════════════════════════════════════════

from flask import Flask

def create_app(config_name="default"):
    app = Flask(__name__)
    app.config["TESTING"] = (config_name == "testing")

    # Init extensions — each gets the app injected here
    print(f"\nCreating app with config={config_name!r} …")
    db.init_app(app)
    login_manager.init_app(app)

    return app

app1 = create_app("development")
app2 = create_app("testing")

print("\n✅  Each app has its own extension binding:")
print(f"   app1 extensions: {list(app1.extensions.keys())}")
print(f"   app2 extensions: {list(app2.extensions.keys())}")
print(f"   Same db object? {db.app is app2}  (bound to whichever was last initialised)")


In [ ]:
# ── Project structure as ASCII tree ──────────────────────────────────

tree = """
my_flask_app/
├── app/
│   ├── __init__.py          ← create_app() lives here
│   ├── extensions.py        ← db, login_manager, mail … (no app yet)
│   │
│   ├── auth/
│   │   ├── __init__.py      ← exposes auth_bp
│   │   ├── routes.py        ← @auth_bp.route(...)
│   │   ├── forms.py
│   │   └── templates/
│   │       └── auth/
│   │           ├── login.html
│   │           └── register.html
│   │
│   ├── blog/
│   │   ├── __init__.py      ← exposes blog_bp
│   │   ├── routes.py
│   │   ├── models.py
│   │   └── templates/
│   │       └── blog/
│   │           ├── index.html
│   │           └── post.html
│   │
│   ├── admin/
│   │   ├── __init__.py      ← exposes admin_bp
│   │   ├── routes.py
│   │   └── templates/
│   │       └── admin/
│   │           └── dashboard.html
│   │
│   ├── static/              ← shared CSS / JS / images
│   └── templates/
│       └── base.html        ← shared base template
│
├── config.py                ← Config, DevelopmentConfig, TestingConfig …
├── wsgi.py                  ← entry point: app = create_app()
├── requirements.txt
└── tests/
    ├── conftest.py          ← pytest fixture: app = create_app("testing")
    ├── test_auth.py
    └── test_blog.py
"""

print(tree)


In [ ]:
# ── Blueprint-owned templates ────────────────────────────────────────
# Each blueprint can carry its own templates folder.
# Flask looks in: app/templates  THEN  <blueprint>/templates

from flask import Blueprint, Flask
import os, tempfile, textwrap

# Simulate a blueprint that has a private templates folder
with tempfile.TemporaryDirectory() as tmpdir:
    # Create folder structure
    tmpl_dir = os.path.join(tmpdir, "templates", "auth")
    os.makedirs(tmpl_dir)

    # Write a tiny Jinja2 template
    with open(os.path.join(tmpl_dir, "login.html"), "w") as f:
        f.write("<h1>Login — from auth blueprint templates</h1>\n{{ message }}")

    # Blueprint with template_folder pointing at our tmp dir
    auth_bp = Blueprint(
        "auth",
        __name__,
        url_prefix="/auth",
        template_folder=os.path.join(tmpdir, "templates"),
    )

    @auth_bp.route("/login")
    def login():
        from flask import render_template
        return render_template("auth/login.html", message="Welcome!")

    app = Flask(__name__)
    app.register_blueprint(auth_bp)

    with app.test_client() as c:
        resp = c.get("/auth/login")
        print("Template rendered by blueprint:")
        print(" ", resp.data.decode())

print("""
Key points:
  ● template_folder is relative to the blueprint's root_path
  ● Use a sub-folder (auth/) inside templates/ to avoid name collisions
  ● Blueprint templates are found AFTER the app's own templates/
    (the app templates always win — useful for overrides)
""")


In [ ]:
# ── url_for with blueprint namespaces — live demo ─────────────────────
from flask import Flask, Blueprint, url_for

auth_bp = Blueprint("auth", __name__, url_prefix="/auth")
blog_bp = Blueprint("blog", __name__, url_prefix="/blog")

@auth_bp.route("/login")
def login(): return "login"

@auth_bp.route("/register")
def register(): return "register"

@blog_bp.route("/")
def index(): return "blog index"

@blog_bp.route("/<int:post_id>")
def post_detail(post_id): return f"post {post_id}"

app = Flask(__name__)
app.register_blueprint(auth_bp)
app.register_blueprint(blog_bp)

with app.test_request_context():
    print("=== url_for with blueprint namespacing ===")
    print()

    examples = [
        ("auth.login",                     {}),
        ("auth.register",                  {}),
        ("blog.index",                     {}),
        ("blog.post_detail",               {"post_id": 42}),
        ("auth.login",                     {"next": "/dashboard"}),
    ]

    for endpoint, kwargs in examples:
        url = url_for(endpoint, **kwargs)
        print(f"  url_for({endpoint!r}{(', ' + str(kwargs)) if kwargs else ''}) → {url!r}")

    print()
    print("Rule of thumb:")
    print("  ✅  url_for('auth.login')   ← blueprint_name.function_name")
    print("  ❌  url_for('login')        ← only works WITHOUT blueprints")


## ⚖️ Comparison: Single-file vs Blueprints

The table below summarises when each approach makes sense.

| Criterion | Single file | Blueprints + Factory |
|---|---|---|
| Project size | Small (< ~200 lines) | Medium → Large |
| Team size | Solo | 2+ developers |
| Testability | Hard (global `app`) | Easy (`create_app("testing")`) |
| Config flexibility | One config | Multiple configs |
| Route namespacing | Flat | Namespaced (`auth.login`) |
| Template organisation | One folder | Per-blueprint folders |
| Learning curve | Low | Slightly higher |

In [ ]:
# ── Side-by-side: same feature in both styles ────────────────────────

single_file_code = """
# ── SINGLE FILE (app.py) ────────────────────────────────────────────

from flask import Flask, render_template, redirect, url_for, flash

app = Flask(__name__)
app.secret_key = "hardcoded-bad"      # ← config mixed into code

@app.route("/login", methods=["GET","POST"])
def login():                           # ← lives in global namespace
    ...
    return redirect(url_for("dashboard"))  # ← no prefix

@app.route("/dashboard")
def dashboard():
    ...

if __name__ == "__main__":
    app.run(debug=True)               # ← debug flag hardcoded
"""

blueprint_code = """
# ── WITH BLUEPRINTS ──────────────────────────────────────────────────

# auth/routes.py
from flask import Blueprint, render_template, redirect, url_for, flash
auth_bp = Blueprint("auth", __name__, url_prefix="/auth")

@auth_bp.route("/login", methods=["GET","POST"])
def login():                           # ← lives in auth namespace
    ...
    return redirect(url_for("main.dashboard"))   # ← explicit namespace

# main/routes.py
main_bp = Blueprint("main", __name__)

@main_bp.route("/dashboard")
def dashboard():
    ...

# app/__init__.py
def create_app(config="default"):
    app = Flask(__name__)
    app.config.from_object(config)     # ← config from object
    app.register_blueprint(auth_bp)
    app.register_blueprint(main_bp)
    return app

# wsgi.py
# app = create_app()                  # ← no debug flag here
"""

print("SINGLE FILE:")
print(single_file_code)
print("\nBLUEPRINTS + FACTORY:")
print(blueprint_code)


In [ ]:
# ── Comparison verdict table ─────────────────────────────────────────

rows = [
    ("Prototyping a weekend project",       "Single file",  "Fast, no overhead"),
    ("Hackathon (solo, 48 h)",              "Single file",  "Speed matters"),
    ("Production API (team of 3)",          "Blueprints",   "Parallel dev, CI/CD"),
    ("SaaS with auth + billing + admin",    "Blueprints",   "Mandatory separation"),
    ("Open-source project (contributors)",  "Blueprints",   "Clear module boundaries"),
    ("Learning Flask basics",               "Single file",  "Less confusion"),
    ("Learning advanced Flask",             "Blueprints",   "Real-world patterns"),
]

col_w = [42, 14, 32]
sep = "+" + "+".join("-"*(w+2) for w in col_w) + "+"
header = "| " + " | ".join(
    ["Scenario".ljust(col_w[0]), "Approach".ljust(col_w[1]), "Reason".ljust(col_w[2])]
) + " |"

print(sep)
print(header)
print(sep)
for scenario, approach, reason in rows:
    icon = "🟦" if approach == "Single file" else "🟩"
    print("| " + " | ".join([
        f"{icon} {scenario}".ljust(col_w[0]),
        approach.ljust(col_w[1]),
        reason.ljust(col_w[2]),
    ]) + " |")
print(sep)


In [ ]:
# ── What If #1: Forgetting url_prefix ────────────────────────────────
# Two blueprints both register /login — Flask silently keeps the FIRST one.

from flask import Flask, Blueprint

auth_bp  = Blueprint("auth",  __name__)   # ← NO url_prefix!
admin_bp = Blueprint("admin", __name__)   # ← NO url_prefix!

@auth_bp.route("/login")
def auth_login():  return "Auth login"

@admin_bp.route("/login")            # ← SAME path!
def admin_login(): return "Admin login"

app = Flask(__name__)
app.register_blueprint(auth_bp)
app.register_blueprint(admin_bp)

client = app.test_client()
resp = client.get("/login")
print("=== Missing url_prefix — route collision ===")
print(f"GET /login → {resp.data.decode()!r}")
print()
print("Only auth_bp's login is reachable — admin_login is silently shadowed.")
print()

# ── The fix ───────────────────────────────────────────────────────────
auth_bp2  = Blueprint("auth2",  __name__, url_prefix="/auth")
admin_bp2 = Blueprint("admin2", __name__, url_prefix="/admin")

@auth_bp2.route("/login")
def auth_login2():  return "Auth login"

@admin_bp2.route("/login")
def admin_login2(): return "Admin login"

app2 = Flask(__name__)
app2.register_blueprint(auth_bp2)
app2.register_blueprint(admin_bp2)

client2 = app2.test_client()
print("=== With url_prefix — no collision ===")
for path in ("/auth/login", "/admin/login"):
    r = client2.get(path)
    print(f"GET {path:20s} → {r.data.decode()!r}")


In [ ]:
# ── What If #2: url_for without blueprint prefix → BuildError ─────────
from flask import Flask, Blueprint, url_for

bp = Blueprint("shop", __name__, url_prefix="/shop")

@bp.route("/cart")
def cart(): return "cart"

app = Flask(__name__)
app.register_blueprint(bp)

print("=== url_for without blueprint prefix ===")
with app.test_request_context():

    # CORRECT
    try:
        url = url_for("shop.cart")
        print(f"✅  url_for('shop.cart')  → {url!r}")
    except Exception as e:
        print(f"❌  {e}")

    # WRONG — forgets the blueprint name
    try:
        url = url_for("cart")
        print(f"✅  url_for('cart')       → {url!r}")
    except Exception as e:
        print(f"❌  url_for('cart') raised {type(e).__name__}: {e}")

print()
print("Lesson: always use  blueprint_name.view_function_name  with url_for.")


In [ ]:
# ── What If #3: Same function name in two blueprints — no conflict ─────
# Blueprints namespace endpoint names, so "auth.index" ≠ "blog.index"

from flask import Flask, Blueprint, url_for

auth_bp = Blueprint("auth", __name__, url_prefix="/auth")
blog_bp = Blueprint("blog", __name__, url_prefix="/blog")

@auth_bp.route("/")
def index():          # ← same name …
    return "Auth home"

@blog_bp.route("/")
def index():          # ← … in a different blueprint → no clash  # noqa: F811
    return "Blog home"

app = Flask(__name__)
app.register_blueprint(auth_bp)
app.register_blueprint(blog_bp)

client = app.test_client()
print("=== Same function name, different blueprints ===")
for path in ("/auth/", "/blog/"):
    r = client.get(path)
    print(f"GET {path:10s} → {r.data.decode()!r}")

print()
with app.test_request_context():
    print("url_for lookups:")
    print(f"  url_for('auth.index') → {url_for('auth.index')!r}")
    print(f"  url_for('blog.index') → {url_for('blog.index')!r}")

print()
print("✅  Blueprint namespacing prevents endpoint collisions even when")
print("   two view functions share the same Python name.")


## 🌍 Real-World Project Layout

The pattern you just learned is used in virtually every production Flask app.

Here are canonical open-source examples to study:

| Project | What to look at |
|---|---|
| [flask-bone](https://github.com/imwilsonxu/fbone) | Classic blueprint + factory layout |
| [cookiecutter-flask](https://github.com/cookiecutter-flask/cookiecutter-flask) | Modern scaffold with all extensions |
| Official [Flask tutorial](https://flask.palletsprojects.com/en/3.0.x/tutorial/) | Step-by-step factory + blueprint |

**What you'll see in every real project:**

```
wsgi.py                 →  app = create_app()
app/__init__.py         →  def create_app(config=None): …
app/extensions.py       →  db = SQLAlchemy(); mail = Mail(); …
app/auth/__init__.py    →  from .routes import auth_bp
app/auth/routes.py      →  @auth_bp.route(…)
config.py               →  class ProductionConfig: …
tests/conftest.py       →  @pytest.fixture def app(): return create_app("testing")
```

In [ ]:
# ── Chapter 8 Cheat Sheet ────────────────────────────────────────────

cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════╗
║              FLASK BLUEPRINTS & APPLICATION FACTORY                 ║
║                        Quick Reference                              ║
╠══════════════════════════════════════════════════════════════════════╣
║  CREATE                                                             ║
║  bp = Blueprint("name", __name__, url_prefix="/prefix")            ║
║                                                                     ║
║  ROUTE                                                              ║
║  @bp.route("/path", methods=["GET","POST"])                        ║
║  def view(): …                                                      ║
║                                                                     ║
║  REGISTER (inside factory)                                          ║
║  app.register_blueprint(bp)                                         ║
║                                                                     ║
║  URL BUILDING                                                       ║
║  url_for("blueprint_name.view_function")                            ║
║  url_for("auth.login")   url_for("blog.index", page=2)             ║
║                                                                     ║
║  FACTORY SKELETON                                                   ║
║  def create_app(config="default"):                                  ║
║      app = Flask(__name__)                                          ║
║      app.config.from_object(config)                                 ║
║      db.init_app(app)             # extensions.py pattern           ║
║      app.register_blueprint(auth_bp)                                ║
║      app.register_blueprint(blog_bp)                                ║
║      return app                                                     ║
║                                                                     ║
║  EXTENSIONS.PY PATTERN                                              ║
║  db = SQLAlchemy()        ← no app                                  ║
║  db.init_app(app)         ← inside create_app()                     ║
║                                                                     ║
║  BLUEPRINT TEMPLATES                                                ║
║  Blueprint("bp", __name__, template_folder="templates")            ║
║  Keep templates in  templates/<bp_name>/  sub-folder               ║
║                                                                     ║
║  COMMON GOTCHAS                                                     ║
║  ❌ Forgetting url_prefix  → route collision, silent shadow         ║
║  ❌ url_for("login")       → BuildError (should be "auth.login")   ║
║  ✅ Same fn name in 2 BPs → fine, namespaced as auth.x / blog.x    ║
╚══════════════════════════════════════════════════════════════════════╝
"""

print(cheat_sheet)


## 🏋️ Practice Prompts

Work through these to solidify your understanding:

1. **Refactor** — Take a 200-line single-file Flask app and split it into at least two blueprints (`auth` and `main`). Verify nothing breaks by running the tests.

2. **Factory configs** — Add a `TestingConfig` class in `config.py` and write a `pytest` fixture that calls `create_app(TestingConfig)`. Run a route test using the fixture.

3. **Blueprint templates** — Give the `blog` blueprint its own `templates/blog/` folder. Render a `blog/index.html` template from a blueprint route. Override it from the app-level `templates/` folder.

4. **url_for audit** — Search an existing project for all `url_for(` calls. Verify every one uses the `blueprint.view` format. Fix any that don't.

5. **Error handlers** — Register a `404` error handler on a blueprint (`@bp.app_errorhandler(404)`) and confirm it fires for any 404 in the whole app, not just that blueprint's routes.

6. **extensions.py** — Add `Flask-Mail` to the project using the `extensions.py` pattern. Initialise it in the factory and send a test email from a blueprint route using `current_app.extensions`.

---
*Tip: the official Flask docs [Application Factories](https://flask.palletsprojects.com/en/3.0.x/patterns/appfactories/) and [Blueprints](https://flask.palletsprojects.com/en/3.0.x/blueprints/) pages are excellent companions to this chapter.*

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../2.%20working_with_data/7.%20database_migrations_with_flask_migrate.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 7: Migrations</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='9.%20configuration_management.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 9: Configuration →</a>
</div>